In [40]:
import os
from google import genai
import pandas as pd
import random
import json
import ast

In [2]:
def call_gemini(user_prompt: str, system_prompt: str = None):
    api_key = os.getenv("google_ai")
    client = genai.Client(api_key=api_key) 
    model = "gemini-2.0-flash"
    if system_prompt:
        response = client.models.generate_content( model="gemini-2.5-flash", config=genai.types.GenerateContentConfig( system_instruction=system_prompt ), contents=user_prompt )
    else: response = client.models.generate_content(model=model, contents=user_prompt)
    return response

# Part 1: Sentiment Analysis

In [14]:
sentimentAnalysis = pd.read_csv('train.csv', encoding = 'latin1')

random.seed(42)

indices = []
for i in range(25):
    index = random.randint(0, len(sentimentAnalysis)-1)
    indices.append(index)

indices.sort()

sentimentAnalysisTrunc = sentimentAnalysis.iloc[indices]

In [4]:
item = sentimentAnalysisTrunc['text'] 
prompt = \
    f"""
    Your task is to analyze a string of text and determine its sentiment.
    I will pass a Pandas series object containing 25 strings.
    For each string, classify its sentiment as either "positive", "negative", or "neutral".
    Your only options are "positive", "negative", or "neutral".
    Respond with just one word in each instance.
    For example, if the string is "This is the best day ever", your response should be "positive".
    Compile your responses as a list of 25 words in the same order as the input strings.
    
    String to process: {item}
    """
chat_response = call_gemini(prompt)

response_text = chat_response.candidates[0].content.parts[0].text

In [6]:
dataList = json.loads(response_text.strip())
dataSeries = pd.Series(dataList)

# Reset indices to ensure alignment
actual_sentiment = sentimentAnalysisTrunc['sentiment'].reset_index(drop=True)
predicted_sentiment = dataSeries.reset_index(drop=True)

# Create the DataFrame
comparisonDF = pd.DataFrame({
    'text': sentimentAnalysisTrunc['text'].reset_index(drop=True),
    'Actual Sentiment': actual_sentiment,
    'Predicted Sentiment': predicted_sentiment,
    'correct' : actual_sentiment == predicted_sentiment
})
comparisonDF

,text,Actual Sentiment,Predicted Sentiment,correct
0,well hit me and we can see...it depends then,neutral,neutral,True
1,"You know, your updates are really amusing. H...",positive,positive,True
2,_Aid16 Goodnight!,neutral,neutral,True
3,tweeten maar,neutral,neutral,True
4,going through security already miss my baby.,negative,negative,True
5,"thanks i have to finish schoolwork today, no...",positive,negative,False
6,Listening to Metal Shop at Mooneys!! All is good,positive,positive,True
7,i know. But.,neutral,neutral,True
8,lol nah no swine flu for me bro. lmao what`s ...,neutral,neutral,True
9,hey is there a way u can make a somatic theme...,neutral,neutral,True


In [7]:
sum(comparisonDF['correct']) / len(comparisonDF)

0.76

The LLM achieved a 76% accuracy analyzing the sentiment of the 25 randomly selected tweets from the dataset. The model seemed to do really well on shorter phrases. It struggled on longer phrases though, especially when there are two sentences in the phrase and the first sentence taken alone would have a different sentiment from the whole input text. I could likely improve model performance by including an example like this in my prompt and telling the model to watch out for this situation.

In [9]:
chat_response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=76,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=76
    ),
  ],
  prompt_token_count=564,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=564
    ),
  ],
  total_token_count=640
)

This query used 564 input tokens and produced 76 output tokens.

# Part 2: Misclassified Articles

In [10]:
headlines = pd.read_csv('misclassified_headlines.csv', encoding = 'latin1')

In [19]:
categories = headlines['category'].unique()

random.seed(42)

indices = []
for i in range(25):
    index = random.randint(0, len(headlines)-1)
    indices.append(index)

indices.sort()

headlinesTrunc = headlines.iloc[indices]
headlinesTrunc

,category,Predicted,link,headline,category_original,short_description,authors,date
204,STYLE & BEAUTY,PARENTING,https://www.huffingtonpost.com/entry/children-...,Children Are Changing the World!,STYLE & BEAUTY,"They were born with Google, computers and have...","Vicky Tiel, Contributor\nDesigner/Author",2014-01-08
217,"ARTS, CULTURE, & ENTERTAINMENT",POLITICS,https://www.huffingtonpost.com/entry/best-and-...,"Nearly Everything That Happened In 2016, In On...","ARTS, CULTURE, & ENTERTAINMENT","From ""Stranger Things"" and Bernie Sanders to t...",Katherine Brooks,2016-12-19
244,ENVIRONMENT,SCIENCE,https://www.huffingtonpost.com/entry/dry-hop-r...,Drinking Beer Could Help Save This Adorable Re...,ENVIRONMENT,NaN,Kim Bellware,2014-04-25
260,PARENTING,"ARTS, CULTURE, & ENTERTAINMENT",https://www.huffingtonpost.com/entry/wes-the-e...,Wes the Extra Ordinary,PARENTING,"During his last moments, when it looked as tho...","Ashbey Riley, ContributorWriter, Filmmaker, En...",2015-04-07
712,PARENTING,EDUCATION,https://www.huffingtonpost.com/entry/stacy-kol...,Cafeteria Worker Resigns Over School District'...,PARENTING,"""I will never forget the look on his face and ...",Rebecca Shapiro,2016-09-21
767,ENVIRONMENT,WELLNESS,https://www.huffingtonpost.com/entry/stop-reck...,Stop Reckless Drilling: A New Year's Resolutio...,ENVIRONMENT,"This year, let's make a New Year's resolution ...","Vikki N. Spruill, Contributor\nCEO, Ocean Cons...",2013-01-12
839,INTERNATIONAL NEWS,ENVIRONMENT,https://www.huffingtonpost.com/entry/canada-ar...,Canada Designates Its Second And Largest Arcti...,INTERNATIONAL NEWS,"Several Inuvialuit organizations, including th...","Hannah Hoag, Arctic Deeply",2016-12-02
912,"ARTS, CULTURE, & ENTERTAINMENT",SCIENCE,https://www.huffpost.com/entry/matthew-mcconau...,Matthew McConaughey Predicts The Bizarre Way H...,"ARTS, CULTURE, & ENTERTAINMENT","The actor has a very different idea of a ""natu...",Ed Mazza,2021-10-08
1143,TRAVEL,FOOD & DRINK,https://www.huffingtonpost.com/entry/lodi-is-f...,Lodi is for (Wine) Lovers,TRAVEL,"A more economical choice, the Lodi Holiday Inn...","offMetro, Contributor\nGet out of town, Car Op...",2013-10-04
1628,FOOD & DRINK,STYLE & BEAUTY,https://www.huffingtonpost.com/entry/starbucks...,Starbucks Bouncer Video 'Are You On The List' ...,FOOD & DRINK,"This might be the future of Starbucks, so you'...",Alison Spiegel,2014-03-28


In [27]:
item = headlinesTrunc['headline']
prompt = \
    f"""
    Your task is to analyze a string of text and determine its category.
    I will pass a Pandas series object containing 25 strings.
    For each string, classify its category as one of the following: {', '.join(categories)}.
    Respond with just the category label in each instance.
    For example, if the string is "BPA Linked With Potential Heart And Kidney Problems", your response should be "WELLNESS".
    Compile your responses as a Python list of 25 words or phrases in the same order as the input strings.
    Only return the list, no other text. The first character in your output should be "[", and there should be no "\n" characters in your response.
    
    String to process: {item}
    """
chat_response = call_gemini(prompt)

response_text = chat_response.candidates[0].content.parts[0].text

In [28]:
response_text

"['PARENTING', 'POLITICS', 'ENVIRONMENT', 'ARTS, CULTURE, & ENTERTAINMENT', 'EDUCATION', 'ENVIRONMENT', 'ENVIRONMENT', 'ARTS, CULTURE, & ENTERTAINMENT', 'FOOD & DRINK', 'ARTS, CULTURE, & ENTERTAINMENT', 'ARTS, CULTURE, & ENTERTAINMENT', 'STYLE & BEAUTY', 'ENVIRONMENT', 'QUEER VOICES', 'TRAVEL', 'POLITICS', 'TRAVEL', 'SPORTS', 'INTERNATIONAL NEWS', 'ARTS, CULTURE, & ENTERTAINMENT', 'WELLNESS', 'SPORTS', 'WELLNESS', 'ARTS, CULTURE, & ENTERTAINMENT', 'TECH']\n"

In [41]:
response_list = ast.literal_eval(response_text)
dataSeries = pd.Series(response_list)

# Reset indices to ensure alignment
actual_category = headlinesTrunc['category'].reset_index(drop=True)
predicted_category = dataSeries.reset_index(drop=True)

# Create the DataFrame
comparisonDF = pd.DataFrame({
    'headline': headlinesTrunc['headline'].reset_index(drop=True),
    'Actual Category': actual_category,
    'Predicted Category': predicted_category,
    'correct' : actual_category == predicted_category
})
comparisonDF

,headline,Actual Category,Predicted Category,correct
0,Children Are Changing the World!,STYLE & BEAUTY,PARENTING,False
1,"Nearly Everything That Happened In 2016, In On...","ARTS, CULTURE, & ENTERTAINMENT",POLITICS,False
2,Drinking Beer Could Help Save This Adorable Re...,ENVIRONMENT,ENVIRONMENT,True
3,Wes the Extra Ordinary,PARENTING,"ARTS, CULTURE, & ENTERTAINMENT",False
4,Cafeteria Worker Resigns Over School District'...,PARENTING,EDUCATION,False
5,Stop Reckless Drilling: A New Year's Resolutio...,ENVIRONMENT,ENVIRONMENT,True
6,Canada Designates Its Second And Largest Arcti...,INTERNATIONAL NEWS,ENVIRONMENT,False
7,Matthew McConaughey Predicts The Bizarre Way H...,"ARTS, CULTURE, & ENTERTAINMENT","ARTS, CULTURE, & ENTERTAINMENT",True
8,Lodi is for (Wine) Lovers,TRAVEL,FOOD & DRINK,False
9,Starbucks Bouncer Video 'Are You On The List' ...,FOOD & DRINK,"ARTS, CULTURE, & ENTERTAINMENT",False


In [42]:
sum(comparisonDF['correct']) / len(comparisonDF)

0.52

This time, the LLM only achieved an accuracy of 52%. It's not surprising that the accuracy here is lower because these were observations that were more difficult for the other model to predict as well. Many of the examples that the LLM was unable to classify are examples that I would have a hard time with as well, such as an article about Drew Brees being a Food & Drink article, or an article about wine being a Travel article instead of Food & Drink. I think one thing that I could have done to help my model's performance was to include in my prompt that if the headline had another country's name in it, it is most likely an International News article.

In [43]:
chat_response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=119,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=119
    ),
  ],
  prompt_token_count=596,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=596
    ),
  ],
  total_token_count=715
)

The model used 596 tokens to process this prompt, and output 119 tokens.